In [67]:
import sys
import time
from selenium import webdriver
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import uuid


In [80]:
driver = webdriver.Chrome()
driver.get("https://results.hyrox.com/season-8/")
time.sleep(2)
soup = BeautifulSoup(driver.page_source, "html.parser")

In [ ]:
wait = WebDriverWait(driver, 10)

# This is where we could implement a for loop to select the different cities etc.
selected_city = "2026 Amsterdam"
selected_hyrox_type = "HYROX - Overall"
selected_gender = "M"

# Select the city
city_dropdown = Select(
    wait.until(EC.presence_of_element_located(
        (By.ID, "default-lists-event_main_group")
    ))
)
# get a list of all the cities
cities = [
    option.get_attribute("value")
    for option in city_dropdown.options
]
#To be changed later to check different cities, does nothing yet and may not be right
cities_to_check = ["2026 Amsterdam", "2026 Rotterdam", "2026 Heerenveen", "2026 Mechelen", "2026 Gent", "2026 Maastricht", "2026 utrecht"]

#actually click amsterdam
city_dropdown.select_by_value(selected_city)

#allow page to update, the other dropdowns get reloaded after picking a city
time.sleep(2)

# We want only the overall hyrox so get the different types of events
# have to find it again here because the page changed after picking the city
event_dropdown = Select(
    wait.until(EC.presence_of_element_located(
        (By.ID, "default-lists-event")
    ))
)
# Create a list of the different types, may be useful later
event_types = [
    option.get_attribute("value")
    for option in event_dropdown.options
]
# use the visible label HYROX overall label and select it
event_dropdown.select_by_visible_text(selected_hyrox_type)

# The same code but this time for the gender filter
gender_dropdown = Select(
    wait.until(EC.presence_of_element_located(
        (By.ID, "default-lists-sex")
    ))
)
# Choose just men for now to check if everything works
gender_dropdown.select_by_value(selected_gender)

# Same code but for the number of rows per page
results_per_page = Select(
    wait.until(EC.presence_of_element_located((By.ID, "default-num_results")))
)
#Select 100 results per page to make scraping faster
results_per_page.select_by_value("100")

#give the page a moment to settle after all the filter changes
time.sleep(2)

#wait until the show results button can actually be clicked --> earlier the button sometimes wouldn't click
submit_button = wait.until(EC.element_to_be_clickable((By.ID, "default-submit")))

#click the show results button which is named default submit in the inspect tool
driver.find_element(By.ID, "default-submit").click()

In [ ]:
import uuid
from urllib.parse import urljoin

soup = BeautifulSoup(driver.page_source, "html.parser")

rows = soup.select(
    'div[data-sex="M"] ul.list-group > li.list-group-item:not(.list-group-header)'
)
print("Result rows found:", len(rows))

collection_id = str(uuid.uuid4())
results_ranking_page = []

for row in rows:
    flag = row.find("img", class_="nation__icon")
    abbreviation = flag.get("alt") if flag else None

    link_tag = row.find("a", href=True)
    personal_link = urljoin(driver.current_url, link_tag["href"]) if link_tag else None #we need the full link instead of the relative link otherwise it did not woerk

    age_field = row.find("div", class_="type-age_class")
    age_group = age_field.get_text(" ", strip=True).replace("Age Group", "").strip() if age_field else None

    time_field = row.find("div", class_="type-time")
    total_time = time_field.get_text(" ", strip=True).replace("Total", "").strip() if time_field else None

    results_ranking_page.append({
        "collection_id": collection_id,
        "record_id": str(uuid.uuid4()),
        "nationality_athlete": abbreviation,
        "city_of_event": selected_city,
        "age_group": age_group,
        "total_time": total_time,
        "personal_link": personal_link
    })

print(results_ranking_page[:5])


Result rows found: 100
[{'collection_id': 'b70373bd-e4eb-453f-935b-d6ac6086c095', 'record_id': '1f649c4b-1246-48c3-a552-a9b24fbcae53', 'nationality_athlete': 'GER', 'city_of_event': '2026 Amsterdam', 'age_group': '35-39', 'total_time': '00:53:43', 'personal_link': 'https://results.hyrox.com/season-8/?content=detail&fpid=list&pid=list&idp=LR3MS4JI45806E_AMS&lang=EN_CAP&event=H_AMS26_OVERALL&num_results=100&pidp=ranking_nav&ranking=time_finish_netto&search%5Bsex%5D=M&search%5Bage_class%5D=%25&search%5Bnation%5D=%25&search_event=H_AMS26_OVERALL'}, {'collection_id': 'b70373bd-e4eb-453f-935b-d6ac6086c095', 'record_id': '33ff93e2-476b-4d55-9e0a-cde6ef032171', 'nationality_athlete': 'NED', 'city_of_event': '2026 Amsterdam', 'age_group': '40-44', 'total_time': '00:54:55', 'personal_link': 'https://results.hyrox.com/season-8/?content=detail&fpid=list&pid=list&idp=LR3MS4JI456F3F_AMS&lang=EN_CAP&event=H_AMS26_OVERALL&num_results=100&pidp=ranking_nav&ranking=time_finish_netto&search%5Bsex%5D=M&sea

Part two: clicking on each participant's link (Works :))


In [81]:
import time
from selenium.webdriver.support.ui import WebDriverWait

# Test navigation through 5 athlete pages
results_detail_page = []

for i, athlete in enumerate(results_ranking_page, start=1):

    personal_link = athlete.get("personal_link")


    if not personal_link:
        continue

    # Navigate to the athlete's page in Chrome
    driver.get(personal_link)

    # Wait for the document to load
    WebDriverWait(driver, 15).until(
        lambda d: d.execute_script(
            "return document.readyState") == "complete")

    print(f"Opened athlete {i}: {driver.current_url}")

    # Keep the page visible for inspection
    time.sleep(1)

    personal_soup = BeautifulSoup(
    driver.page_source,
    "html.parser")

    results_detail_page.append({
        "collection_id": athlete["collection_id"],
        "record_id": athlete["record_id"],
        "html": str(personal_soup)})

    print("Total personal pages saved:", len(results_detail_page))



Opened athlete 1: https://results.hyrox.com/season-8/?content=detail&fpid=list&pid=list&idp=LR3MS4JI45806E_AMS&lang=EN_CAP&event=H_AMS26_OVERALL&num_results=100&pidp=ranking_nav&ranking=time_finish_netto&search%5Bsex%5D=M&search%5Bage_class%5D=%25&search%5Bnation%5D=%25&search_event=H_AMS26_OVERALL
Total personal pages saved: 1
Opened athlete 2: https://results.hyrox.com/season-8/?content=detail&fpid=list&pid=list&idp=LR3MS4JI456F3F_AMS&lang=EN_CAP&event=H_AMS26_OVERALL&num_results=100&pidp=ranking_nav&ranking=time_finish_netto&search%5Bsex%5D=M&search%5Bage_class%5D=%25&search%5Bnation%5D=%25&search_event=H_AMS26_OVERALL
Total personal pages saved: 2
Opened athlete 3: https://results.hyrox.com/season-8/?content=detail&fpid=list&pid=list&idp=LR3MS4JI458A66_AMS&lang=EN_CAP&event=H_AMS26_OVERALL&num_results=100&pidp=ranking_nav&ranking=time_finish_netto&search%5Bsex%5D=M&search%5Bage_class%5D=%25&search%5Bnation%5D=%25&search_event=H_AMS26_OVERALL
Total personal pages saved: 3
Opened ath

Step 3: scraping from the personal page (Workout summary) (scrapes the times double so not done yet)

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup

results_workout_summary = []

wanted_classes = ([f"f-time_{i}" for i in range (11,19)]
            + ["f-time_49", "f-time_50", "f-time_60"])

for athlete in results_detail_page:
    personal_soup = BeautifulSoup(athlete["html"], "html.parser")

# Select the workout summary table
    table = personal_soup.select_one("#detail-box-other table.table")

    if table is None:
        print(f"No workout summary found for athlete {i}") 

    else: 
        for row in table.select("tbody tr"):
            row_classes = row.get("class", [])

            if any(cls in row_classes for cls in wanted_classes):
                cells = row.find_all(["th", "td"], recursive = False)
            if len(cells) == 3: 
                results_workout_summary.append({
                    "workout": cells[0].get_text(strip=True),
                    "split_time": cells[1].get_text(strip=True),
                    "place": cells[2].get_text(strip=True)
                })
       

df_workout= pd.DataFrame(results_workout_summary)
display(df_workout.head(15))



,workout,split_time,place
0,Best Run Lap,00:03:36,15
1,1000m SkiErg,00:03:51,5
2,1000m SkiErg,00:03:51,5
3,50m Sled Push,00:02:04,6
4,50m Sled Push,00:02:04,6
5,50m Sled Pull,00:02:58,6
6,50m Sled Pull,00:02:58,6
7,80m Burpee Broad Jump,00:02:24,2
8,80m Burpee Broad Jump,00:02:24,2
9,1000m Row,00:03:59,1


Step 4: Scraping the judging decisions page (not done yet)

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup

judging_decisions = []

for athlete in results_detail_page:
    personal_soup = BeautifulSoup(athlete["html"], "html.parser")

heading = personal_soup.find("h3", string= "Judging Decisions")
judging_table = heading.find_next("table") if heading else None

if judging_table is not None:
    for row in judging_table.select("tbody tr"):
        cells = row.find_all(["th", "td"], recursive=False)
        if len(cells) >= 2:
            judging_decisions.append({
                "decision": cells[0].get_text(strip=True),
                "value": (None if value in ["–", "-", ""] else value)
            })

df_judging = pd.DataFrame(judging_decisions)
display(df_judging.head(15))